# 01D — ETL Nguon UCI Student Performance -> DataMart Chuan Hoa

**Dataset:** UCI Student Performance (Cortez & Silva, 2008)
**Input:** `data/raw/uci_student_performance/student-mat.csv` + `student-por.csv`
**Output:** `data/processed/uci_perf_datamart.csv`
**Vai tro OOD:** Tap **TEST DOC LAP 2** — Blind Predict.

---

### So do cot nguon -> DataMart

| Cot DataMart | Cot nguon | Ghi chu |
|---|---|---|
| `admission_score_norm` | `Medu`, `Fedu` | `(Medu+Fedu)/8.0` — proxy hoc van phu huynh |
| `credits_registered` | — | `6` (2 mon x 3 TC) |
| `credit_completion_ratio` | `G3` | `1.0 if G3 >= 10` |
| `cumulative_credit_debt` | `failures` | `failures x 6` |
| `gpa_current_semester` | `G3` | Piecewise |
| `cpa_cumulative_score` | `G1,G2,G3` | avg -> Piecewise |
| `training_point_norm` | `absences` | `(1-abs/93).clip(0,1)` |
| `lms_*` | Khong co | **NaN** |


In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.4f}'.format)

RAW_DIR       = 'data/raw/'       if os.path.basename(os.getcwd()) != 'notebooks' else '../data/raw/'
PROCESSED_DIR = 'data/processed/' if os.path.basename(os.getcwd()) != 'notebooks' else '../data/processed/'
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"RAW_DIR: {os.path.abspath(RAW_DIR)}")
print(f"PROCESSED_DIR: {os.path.abspath(PROCESSED_DIR)}")


## 1. Load & Kiem Tra Du Lieu Tho

In [ ]:
df_mat = pd.read_csv(RAW_DIR + 'uci_student_performance/student-mat.csv', sep=';')
df_por = pd.read_csv(RAW_DIR + 'uci_student_performance/student-por.csv', sep=';')
df = pd.concat([df_mat, df_por], ignore_index=True)

print(f"Math shape: {df_mat.shape} | Por shape: {df_por.shape}")
print(f"Combined shape: {df.shape}")
print()
print("=== Thong ke G1, G2, G3 (thang 0-20) ===")
print(df[['G1', 'G2', 'G3']].describe())
print()
print("=== Phan phoi failures ===")
print(df['failures'].value_counts().sort_index())
print()
print("=== Hoc van phu huynh (Medu, Fedu: 0-4) ===")
print(f"Medu: {sorted(df['Medu'].unique())}")
print(f"Fedu: {sorted(df['Fedu'].unique())}")
print()
print("=== absences (vang mat) ===")
print(df['absences'].describe())
print(f"Max absences: {df['absences'].max()} (dung lam mau so chuan hoa)")


## 2. Ham Piecewise GPA Alignment

In [ ]:
def map_uci_gpa(g):
    """Piecewise: UCI 0-20 -> GPA 0-4, qua mon (10) -> 2.0"""
    if pd.isna(g): return 0.0
    if g < 10:
        return (g / 10.0) * 2.0
    return 2.0 + ((g - 10.0) / 10.0) * 2.0

test_grades = [0, 5, 10, 12, 14, 17, 20]
print("Grade -> GPA (Piecewise):")
for g in test_grades:
    marker = " <- Nguong qua mon" if g == 10 else ""
    print(f"  grade={g:2d} -> GPA={map_uci_gpa(g):.3f}{marker}")


## 3. Anh Xa -> 22 Cot DataMart

In [ ]:
g1 = df['G1']
g2 = df['G2']
g3 = df['G3']

out = pd.DataFrame()

out['student_code']        = 'UP_' + df.index.astype(str)
out['academic_year_order'] = 1

# Admission: proxy tu hoc van phu huynh (Medu+Fedu in [0,8] -> [0,1])
# Co so: nghien cuu giao duc xac nhan hoc van phu huynh tuong quan voi nang luc dau vao
out['admission_score_norm'] = ((df['Medu'] + df['Fedu']) / 8.0).clip(0, 1)

out['priority_flag'] = 0  # UCI Perf khong co cot disability

out['credits_registered']      = 6
out['credit_completion_ratio'] = np.where(g3 >= 10, 1.0, 0.0)
out['cumulative_credit_debt']  = df['failures'] * 6

gpa_g3  = g3.apply(map_uci_gpa)
gpa_g1  = g1.apply(map_uci_gpa)
gpa_avg = ((g1 + g2 + g3) / 3.0).apply(map_uci_gpa)

out['gpa_current_semester'] = gpa_g3
out['cpa_cumulative_score'] = gpa_avg
out['cpa_growth_delta']     = gpa_g3 - gpa_g1

out['failed_courses_count'] = df['failures']

# training_point_norm: proxy tu vang mat (max = 93 thuc te dataset)
out['training_point_norm'] = (1.0 - df['absences'] / 93.0).clip(0, 1)

out['tuition_debt_flag'] = 0

# LMS - OOD NaN MASKING
out['lms_total_clicks'] = np.nan
out['lms_active_days']  = np.nan
out['lms_late_submits'] = np.nan
out['lms_avg_quiz']     = np.nan

out['previous_academic_warnings'] = df['failures']
out['risk_score_continuous'] = 0.0

def get_target(g):
    if g < 10: return 3
    if g < 12: return 2
    if g < 14: return 1
    return 0

out['target_risk_level'] = g3.apply(get_target)
out['target_risk_name']  = out['target_risk_level'].map(
    {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Very High'})
out['dataset_source'] = 'UCI_Perf'
out = out.reset_index(drop=True)

print(f"Shape sau ETL: {out.shape}")


## 4. Kiem Tra Chat Luong

In [ ]:
print("=== Phan phoi nhan rui ro ===")
print(out['target_risk_name'].value_counts())
print()

print("=== GPA sau Piecewise ===")
print(out['gpa_current_semester'].describe())
pct_pass = (out['gpa_current_semester'] >= 2.0).mean() * 100
print(f"Ti le GPA >= 2.0: {pct_pass:.1f}%")
print()

print("=== admission_score_norm (tu Medu+Fedu) ===")
print(out['admission_score_norm'].value_counts().sort_index())
print(f"Phuong sai: {out['admission_score_norm'].var():.4f}  <- phai > 0")
print()

print("=== training_point_norm (tu absences) ===")
print(out['training_point_norm'].describe())
print()

print("=== Kiem tra LMS = 100% NaN ===")
for col in ['lms_total_clicks', 'lms_active_days', 'lms_late_submits', 'lms_avg_quiz']:
    pct = out[col].isna().mean() * 100
    status = "OK" if pct == 100 else "FAILED"
    print(f"  {col}: {pct:.1f}% NaN [{status}]")


## 5. Luu File

In [ ]:
out_path = PROCESSED_DIR + 'uci_perf_datamart.csv'
out.to_csv(out_path, index=False)
print(f"Da luu: {out_path}")
print(f"Shape : {out.shape}")
